In [27]:
# RAG advanced evaluation 
# Imports
from pathlib import Path
import json
import re
import time
import numpy as np
import pandas as pd
from tenacity import retry, wait_exponential
import os
import sys
from openai import OpenAI
from dotenv import load_dotenv
from huggingface_hub import login
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from litellm import completion
from multiprocessing import Pool
from tenacity import retry, wait_exponential, stop_after_attempt
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
from pricer.items import Item
from answer import *


In [28]:
# Configuration
GOLDEN_SIZE = 200
RETRIEVAL_K = 10
RANDOM_SEED = 42
EVAL_CHUNKS = 4000
RPM = 30
WORKERS = 2
BATCH_SIZE = 100
FUTURE_TIMEOUT = 120
BATCH_COOLDOWN = 5

In [29]:
# NEW: RAG_eval and preprocessed_db both live in 01_data_pipeline, not in this
# notebook's own folder (06_rag) - these are different directories.
PROJECT_ROOT = Path(notebook_dir).parent.parent   # notebooks/06_rag -> notebooks -> project root
DATA_PIPELINE_DIR = PROJECT_ROOT / "notebooks" / "01_data_pipeline"
GOLDEN_PATH = DATA_PIPELINE_DIR / "RAG_eval" / "golden_dataset_200.json"
PRELIM_GOLDEN_PATH = DATA_PIPELINE_DIR / "RAG_eval" / "preliminary_golden_dataset.json"
DB_NAME = str(DATA_PIPELINE_DIR / "preprocessed_db")  
for p in [GOLDEN_PATH, PRELIM_GOLDEN_PATH, Path(DB_NAME, "chroma.sqlite3")]:
    assert p.exists(), f"Missing: {p}"

In [30]:
# NEW: compare category proportions - final 200 vs preliminary (stratified) 2,100
with open(PRELIM_GOLDEN_PATH, "r", encoding="utf-8") as f:
    prelim = json.load(f)

prelim_dist = pd.Series([x["category"] for x in prelim]).value_counts(normalize=True).sort_index()

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    _final_preview = json.load(f)
final_dist = pd.Series([x["category"] for x in _final_preview]).value_counts(normalize=True).sort_index()

comparison = pd.DataFrame({
    "preliminary (stratified, = KB proportion)": prelim_dist,
    "final_200 (currently used for eval)": final_dist,
}).fillna(0.0)
comparison["diff"] = comparison["final_200 (currently used for eval)"] - comparison["preliminary (stratified, = KB proportion)"]

print(comparison.round(3))
max_diff = comparison["diff"].abs().max()
if max_diff > 0.05:
    print(
        f"\nWARNING: category proportions diverge by up to {max_diff:.1%}. "
        f"The 'Overall' metrics below will be pulled by whichever category is "
        f"over-represented relative to its share of the actual knowledge base "
        f"(commonly 'all beauty', which is only ~5% of the KB but 25% of this set). "
        f"Read the per-category breakdown (further down) as the source of truth, "
        f"not the single overall number."
    )


                  preliminary (stratified, = KB proportion)  \
all beauty                                            0.051   
appliances                                            0.346   
electronics                                           0.316   
home and kitchen                                      0.287   

                  final_200 (currently used for eval)   diff  
all beauty                                      0.050 -0.001  
appliances                                      0.350  0.004  
electronics                                     0.315 -0.001  
home and kitchen                                0.285 -0.002  


In [31]:
# Load the frozen Golden Dataset
with open(GOLDEN_PATH, "r", encoding="utf-8") as file:
    golden_dataset = json.load(file)

print(f"Golden questions: {len(golden_dataset)}")
assert len(golden_dataset) == GOLDEN_SIZE

print(pd.Series([x["category"] for x in golden_dataset]).value_counts())

Golden questions: 200
appliances          70
electronics         63
home and kitchen    57
all beauty          10
Name: count, dtype: int64


In [32]:
# Validate the Golden Dataset
product_ids = [x["product_id"] for x in golden_dataset]

assert len(product_ids) == len(set(product_ids)), "Duplicate product_id found"

required_fields = {
    "product_id",
    "title",
    "category",
    "question",
    "reference_answer",
}

for item in golden_dataset:
    assert required_fields.issubset(item.keys())

print("Golden Dataset validation passed")

Golden Dataset validation passed


In [33]:
# Verify the current Chroma metadata
def get_product_id(doc):
    return doc.metadata.get("product_id")


In [34]:
def retrieved_product_ids(retrieved_docs):
    return [
        doc.metadata.get("product_id")
        for doc in retrieved_docs
        if doc.metadata.get("product_id") is not None
    ]


def recall_at_k(expected_product_id, retrieved_docs, k):
    ids = retrieved_product_ids(retrieved_docs[:k])
    return float(expected_product_id in ids)


def reciprocal_rank(expected_product_id, retrieved_docs):
    ids = retrieved_product_ids(retrieved_docs)
    for rank, product_id in enumerate(ids, start=1):
        if product_id == expected_product_id:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(expected_product_id, retrieved_docs, k):
    ids = retrieved_product_ids(retrieved_docs[:k])
    for rank, product_id in enumerate(ids, start=1):
        if product_id == expected_product_id:
            return 1.0 / np.log2(rank + 1)
    return 0.0

In [35]:
# NEW: relevance = same category + price within tolerance (not exact product_id match)
PRICE_TOLERANCE = 0.30  # a retrieved product counts as "relevant" if within +/-30% of expected price

def is_relevant(expected_category, expected_price, doc_metadata, tolerance=PRICE_TOLERANCE):
    if doc_metadata.get("type") != expected_category:
        return False
    doc_price = doc_metadata.get("price")
    if doc_price is None or expected_price == 0:
        return False
    return abs(doc_price - expected_price) / expected_price <= tolerance


def relevant_flags(expected_category, expected_price, retrieved_docs):
    return [is_relevant(expected_category, expected_price, doc.metadata) for doc in retrieved_docs]


def recall_at_k(expected_category, expected_price, retrieved_docs, k):
    flags = relevant_flags(expected_category, expected_price, retrieved_docs[:k])
    return float(any(flags))


def reciprocal_rank(expected_category, expected_price, retrieved_docs):
    flags = relevant_flags(expected_category, expected_price, retrieved_docs)
    for rank, is_rel in enumerate(flags, start=1):
        if is_rel:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(expected_category, expected_price, retrieved_docs, k):
    flags = relevant_flags(expected_category, expected_price, retrieved_docs[:k])
    for rank, is_rel in enumerate(flags, start=1):
        if is_rel:
            return 1.0 / np.log2(rank + 1)
    return 0.0


In [36]:
def extract_price(text):
    if text is None:
        return None

    text = str(text).replace(",", "")

    # Prefer an explicit dollar amount.
    match = re.search(r"\$\s*(\d+(?:\.\d+)?)", text)
    if match:
        return float(match.group(1))

    # Fall back to a standalone numeric value.
    match = re.search(r"(?<![A-Za-z])(\d+(?:\.\d+)?)(?![A-Za-z])", text)
    if match:
        return float(match.group(1))

    return None


def calculate_price_metrics(generated_answer, reference_answer):
    predicted = extract_price(generated_answer)
    actual = float(reference_answer)

    if predicted is None:
        return {
            "predicted_price": None,
            "absolute_error": None,
            "percentage_error": None,
            "within_10": 0.0,
            "within_20": 0.0,
            "within_30": 0.0,
        }

    absolute_error = abs(predicted - actual)
    percentage_error = (
        absolute_error / abs(actual) * 100
        if actual != 0 else None
    )

    return {
        "predicted_price": predicted,
        "absolute_error": absolute_error,
        "percentage_error": percentage_error,
        "within_10": float(percentage_error <= 10),
        "within_20": float(percentage_error <= 20),
        "within_30": float(percentage_error <= 30),
    }

In [37]:
def evaluate_test(test, k=RETRIEVAL_K):
    question_text = test["question"]

    # Existing RAG pipeline from your agent/notebook.
    rewritten = rewrite_query(question_text)
    retrieved_docs = fetch_context(question_text, rewritten)

    retrieval_metrics = {
        "recall@1": recall_at_k(test["product_id"], retrieved_docs, 1),
        "recall@5": recall_at_k(test["product_id"], retrieved_docs, 5),
        "recall@10": recall_at_k(test["product_id"], retrieved_docs, 10),
        "mrr": reciprocal_rank(test["product_id"], retrieved_docs),
        "ndcg@10": ndcg_at_k(test["product_id"], retrieved_docs, 10),
    }

    messages = make_rag_messages(question_text, retrieved_docs)
    response = completion(
        model=GENERATOR_MODEL,
        messages=messages,
    )

    generated_answer = response.choices[0].message.content
    price_metrics = calculate_price_metrics(
        generated_answer,
        test["reference_answer"],
    )

    return {
        "product_id": test["product_id"],
        "category": test["category"],
        "question": question_text,
        "reference_answer": test["reference_answer"],
        "generated_answer": generated_answer,
        **retrieval_metrics,
        **price_metrics,
    }

In [38]:
# NEW: shared pipeline import (replaces the old duplicated pipeline cells)
sys.path.insert(0, notebook_dir)  # answer.py lives right here in 06_rag
from answer import *


In [39]:
# calculate_price_metrics is evaluation-specific (not part of the answer
# pipeline), so it stays defined here rather than in the shared module.
def calculate_price_metrics(estimate: PriceEstimate, reference_answer):
    actual = float(reference_answer)
    predicted = estimate.estimated_price

    if predicted is None or estimate.confidence == "none":
        return {
            "predicted_price": None,
            "confidence": estimate.confidence,
            "absolute_error": None,
            "percentage_error": None,
            "within_10": 0.0,
            "within_20": 0.0,
            "within_30": 0.0,
        }

    absolute_error = abs(predicted - actual)
    percentage_error = absolute_error / abs(actual) * 100 if actual != 0 else None

    return {
        "predicted_price": predicted,
        "confidence": estimate.confidence,
        "absolute_error": absolute_error,
        "percentage_error": percentage_error,
        "within_10": float(percentage_error is not None and percentage_error <= 10),
        "within_20": float(percentage_error is not None and percentage_error <= 20),
        "within_30": float(percentage_error is not None and percentage_error <= 30),
    }


In [40]:
# evaluate_test - corrected retrieval metrics + structured generation from the shared module
def evaluate_test(test, k=RETRIEVAL_K):
    question_text = test["question"]
    expected_category = test["category"]
    expected_price = float(test["reference_answer"])

    rewritten = rewrite_query(question_text)
    retrieved_docs = fetch_context(question_text, rewritten)

    retrieval_metrics = {
        "recall@1": recall_at_k(expected_category, expected_price, retrieved_docs, 1),
        "recall@5": recall_at_k(expected_category, expected_price, retrieved_docs, 5),
        "recall@10": recall_at_k(expected_category, expected_price, retrieved_docs, 10),
        "mrr": reciprocal_rank(expected_category, expected_price, retrieved_docs),
        "ndcg@10": ndcg_at_k(expected_category, expected_price, retrieved_docs, 10),
    }

    estimate = get_structured_answer(question_text, retrieved_docs)
    price_metrics = calculate_price_metrics(estimate, test["reference_answer"])

    return {
        "product_id": test["product_id"],
        "category": test["category"],
        "question": question_text,
        "reference_answer": test["reference_answer"],
        "generated_answer": estimate.reasoning,
        "matched_category": estimate.matched_category,
        "comparable_products_used": estimate.comparable_products_used,
        **retrieval_metrics,
        **price_metrics,
    }


In [41]:
# NEW: final evaluate_test - corrected retrieval metrics + structured generation
def evaluate_test(test, k=RETRIEVAL_K):
    question_text = test["question"]
    expected_category = test["category"]
    expected_price = float(test["reference_answer"])

    rewritten = rewrite_query(question_text)
    retrieved_docs = fetch_context(question_text, rewritten)

    retrieval_metrics = {
        "recall@1": recall_at_k(expected_category, expected_price, retrieved_docs, 1),
        "recall@5": recall_at_k(expected_category, expected_price, retrieved_docs, 5),
        "recall@10": recall_at_k(expected_category, expected_price, retrieved_docs, 10),
        "mrr": reciprocal_rank(expected_category, expected_price, retrieved_docs),
        "ndcg@10": ndcg_at_k(expected_category, expected_price, retrieved_docs, 10),
    }

    estimate = get_structured_answer(question_text, retrieved_docs)
    price_metrics = calculate_price_metrics(estimate, test["reference_answer"])

    return {
        "product_id": test["product_id"],
        "category": test["category"],
        "question": question_text,
        "reference_answer": test["reference_answer"],
        "generated_answer": estimate.reasoning,
        "matched_category": estimate.matched_category,
        "comparable_products_used": estimate.comparable_products_used,
        **retrieval_metrics,
        **price_metrics,
    }


In [42]:
# Smoke test before running all 200 questions
test_result = evaluate_test(golden_dataset[0])
test_result

{'product_id': 3508,
 'category': 'electronics',
 'question': 'Can you tell me the price of the Kingston ValueRAM 2GB 667MHz DDR2 Non-ECC CL5 DIMM Desktop Memory?',
 'reference_answer': 14.98,
 'generated_answer': 'The Kingston ValueRAM 2GB 800MHz DDR2 Non-ECC CL5 SODIMM is priced at $18.99 and the 2GB DDR2 667MHz non-ECC modules like A-Tech 4GB kit (4x1GB) are around $41 for 4GB (about $10 per GB). Since the target is similar but at 667MHz speed and DIMM form factor (usually slightly cheaper than SODIMM for notebooks), the estimated price for 2GB is approximately $12.',
 'matched_category': True,
 'comparable_products_used': 3,
 'recall@1': 0.0,
 'recall@5': 1.0,
 'recall@10': 1.0,
 'mrr': 0.3333333333333333,
 'ndcg@10': 0.5,
 'predicted_price': 12.0,
 'confidence': 'medium',
 'absolute_error': 2.9800000000000004,
 'percentage_error': 19.893190921228307,
 'within_10': 0.0,
 'within_20': 1.0,
 'within_30': 1.0}

In [43]:
question = golden_dataset[0]["question"]

rewritten = rewrite_query(question)

print("QUESTION:")
print(question)

print("\nREWRITTEN:")
print(rewritten)

retrieved_docs = fetch_context(question, rewritten)

print(f"\nRetrieved: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n===== Rank {i} =====")
    print("Product ID:", doc.metadata.get("product_id"))
    print("Title:", doc.metadata.get("title"))
    print("Category:", doc.metadata.get("type"))
    print("Price:", doc.metadata.get("price"))
    print("Content:")
    print(doc.page_content[:1000])

QUESTION:
Can you tell me the price of the Kingston ValueRAM 2GB 667MHz DDR2 Non-ECC CL5 DIMM Desktop Memory?

REWRITTEN:
Kingston ValueRAM 2GB 667MHz DDR2 Non-ECC CL5 DIMM Desktop Memory

Retrieved: 10

===== Rank 1 =====
Product ID: 6480
Title: MemoryMasters FB-DIMM DDR2 PC2 – 5300 F 667 MHz CL5 ECC 2RX4 240 Pin Memory with Heatspreaders 32GB (8x4GB) 667Mhz HS
Category: electronics
Price: 74.63
Content:
MemoryMasters FB-DIMM DDR2 PC2 – 5300 F 667 MHz CL5 ECC 2RX4 240 Pin Memory with Heatspreaders 32GB (8x4GB) 667Mhz HS

Title: MemoryMasters FB-DIMM DDR2 PC2 – 5300 F 667 MHz CL5 ECC 2RX4 240 Pin Memory with Heatspreaders 32GB (8x4GB) 667Mhz HS

Category: electronics

Description: Title: 32GB DDR2 ECC Fully Buffered Server Memory Kit  
Category: Computer Memory  
Brand: MemoryMasters  
Description: A high-capacity 32GB server memory kit featuring ECC ECC buffered modules with heatspreaders for enhanced stability.  
Details: Includes 8x4GB DDR2 FBDIMM modules operating at 667MHz, compat

In [44]:
# Run the full 200-question evaluation.
# Keep your existing checkpoint/resume implementation here if already available.

results = []

for index, test in enumerate(golden_dataset, start=1):
    try:
        result = evaluate_test(test)
        results.append(result)
        print(f"[{index}/{len(golden_dataset)}] done")
    except Exception as exc:
        print(f"[{index}/{len(golden_dataset)}] failed: {exc}")

results_df = pd.DataFrame(results)


[1/200] done
[2/200] done
[3/200] done
[4/200] done
[5/200] done
[6/200] done
[7/200] done
[8/200] done
[9/200] done
[10/200] done
[11/200] done
[12/200] done
[13/200] done
[14/200] done
[15/200] done
[16/200] done
[17/200] done
[18/200] done
[19/200] done
[20/200] done
[21/200] done
[22/200] done
[23/200] done
[24/200] done
[25/200] done
[26/200] done
[27/200] done
[28/200] done
[29/200] done
[30/200] done
[31/200] done
[32/200] done
[33/200] done
[34/200] done
[35/200] done
[36/200] done
[37/200] done
[38/200] done
[39/200] done
[40/200] done
[41/200] done
[42/200] done
[43/200] done
[44/200] done
[45/200] done
[46/200] done
[47/200] done
[48/200] done
[49/200] done
[50/200] done
[51/200] done
[52/200] done
[53/200] done
[54/200] done
[55/200] done
[56/200] done
[57/200] done
[58/200] done
[59/200] done
[60/200] done
[61/200] done
[62/200] done
[63/200] done
[64/200] done
[65/200] done
[66/200] done
[67/200] done
[68/200] done
[69/200] done
[70/200] done
[71/200] done
[72/200] done
[

In [45]:
# Overall retrieval metrics
retrieval_summary = {
    "Recall@1": results_df["recall@1"].mean(),
    "Recall@5": results_df["recall@5"].mean(),
    "Recall@10": results_df["recall@10"].mean(),
    "MRR": results_df["mrr"].mean(),
    "nDCG@10": results_df["ndcg@10"].mean(),
}

pd.Series(retrieval_summary)

Recall@1     0.410000
Recall@5     0.825000
Recall@10    0.910000
MRR          0.572107
nDCG@10      0.654035
dtype: float64

In [46]:
# Overall price estimation metrics
valid = results_df.dropna(subset=["absolute_error", "percentage_error"]).copy()

answer_summary = {
    "MAE": valid["absolute_error"].mean(),
    "RMSE": np.sqrt(np.mean(valid["absolute_error"] ** 2)),
    "Median AE": valid["absolute_error"].median(),
    "MAPE": valid["percentage_error"].mean(),
    "Within ±10%": valid["within_10"].mean(),
    "Within ±20%": valid["within_20"].mean(),
    "Within ±30%": valid["within_30"].mean(),
    "Answer extraction rate": len(valid) / len(results_df),
}

pd.Series(answer_summary)


MAE                       26.698299
RMSE                      57.124890
Median AE                  8.920000
MAPE                      42.287674
Within ±10%                0.170103
Within ±20%                0.355670
Within ±30%                0.494845
Answer extraction rate     0.970000
dtype: float64

In [47]:
# Category-level analysis
category_results = (
    results_df
    .groupby("category")
    .agg(
        recall_at_10=("recall@10", "mean"),
        mrr=("mrr", "mean"),
        ndcg_at_10=("ndcg@10", "mean"),
        mae=("absolute_error", "mean"),
        within_20=("within_20", "mean"),
    )
)

category_results

,recall_at_10,mrr,ndcg_at_10,mae,within_20
category,,,,,
all beauty,1.000000,0.562619,0.666865,11.332000,0.200000
appliances,0.900000,0.649184,0.711169,28.173333,0.442857
electronics,0.857143,0.500794,0.586310,34.158387,0.317460
home and kitchen,0.964912,0.557937,0.656473,18.950377,0.280702


In [48]:
# Save evaluation results
RESULTS_PATH = GOLDEN_PATH.parent / "rag_evaluation_results_200.csv"
results_df.to_csv(RESULTS_PATH, index=False)

print(f"Saved")


Saved


In [49]:
# NEW: generator versions of the eval loop (yields as it goes, like Old_Path's
# evaluate_all_retrieval()/evaluate_all_answers()), with incremental checkpointing so a
# crash at test #150 doesn't lose the first 149 results.
CHECKPOINT_PATH = GOLDEN_PATH.parent / "eval_checkpoint.jsonl"

def evaluate_all(golden_dataset, resume=True):
    """Yields (test, result, progress) one at a time; appends each result to a
    checkpoint file immediately so partial runs are never lost."""
    done_ids = set()
    if resume and CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                done_ids.add(json.loads(line)["product_id"])
        print(f"Resuming: {len(done_ids)} tests already done, skipping them.")

    total = len(golden_dataset)
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        for index, test in enumerate(golden_dataset, start=1):
            if test["product_id"] in done_ids:
                continue
            try:
                result = evaluate_test(test)
            except Exception as exc:
                result = {"product_id": test["product_id"], "category": test["category"], "error": str(exc)}
            f.write(json.dumps(result) + "\n")
            f.flush()
            yield test, result, index / total


def load_checkpoint_results() -> pd.DataFrame:
    if not CHECKPOINT_PATH.exists():
        return pd.DataFrame()
    rows = []
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if "error" not in row:
                rows.append(row)
    return pd.DataFrame(rows)


In [51]:
# NEW: Gradio dashboard - mirrors Old_Path/Codes/evaluator.py
import gradio as gr
from collections import defaultdict

# Thresholds adapted to this task
RECALL_GREEN, RECALL_AMBER = 0.7, 0.4
MRR_GREEN, MRR_AMBER = 0.6, 0.3
NDCG_GREEN, NDCG_AMBER = 0.6, 0.3
WITHIN20_GREEN, WITHIN20_AMBER = 0.6, 0.35  # "within +/-20% of true price"


def get_color(value, metric_type):
    thresholds = {
        "recall": (RECALL_GREEN, RECALL_AMBER),
        "mrr": (MRR_GREEN, MRR_AMBER),
        "ndcg": (NDCG_GREEN, NDCG_AMBER),
        "within20": (WITHIN20_GREEN, WITHIN20_AMBER),
    }
    green, amber = thresholds[metric_type]
    if value >= green:
        return "green"
    elif value >= amber:
        return "orange"
    return "red"


def format_metric_html(label, value, metric_type, is_percentage=False, is_currency=False):
    color = get_color(value, metric_type) if metric_type in ("recall", "mrr", "ndcg", "within20") else "#333"
    if is_percentage:
        value_str = f"{value:.1%}"
    elif is_currency:
        value_str = f"${value:,.2f}"
    else:
        value_str = f"{value:.4f}"
    return f"""
    <div style="margin: 10px 0; padding: 15px; background-color: #f5f5f5; border-radius: 8px; border-left: 5px solid {color};">
        <div style="font-size: 14px; color: #666; margin-bottom: 5px;">{label}</div>
        <div style="font-size: 28px; font-weight: bold; color: {color};">{value_str}</div>
    </div>
    """


def run_evaluation(progress=gr.Progress()):
    """
    Runs missing tests, then builds the dashboard from ALL checkpoint results.

    This makes the dashboard resume-safe:
    - If 0 tests exist, it evaluates all 200.
    - If 150 tests exist, it evaluates the remaining 50.
    - If all 200 exist, it skips evaluation and loads all 200 results.
    """

    category_recall = defaultdict(list)
    category_mae = defaultdict(list)

    # ---------------------------------------------------------
    # 1. Run missing evaluations
    # ---------------------------------------------------------

    evaluated_new = 0

    for test, result, prog_value in evaluate_all(golden_dataset):

        evaluated_new += 1

        progress(
            prog_value,
            desc=f"Evaluating test {evaluated_new}/{len(golden_dataset)}..."
        )

    # ---------------------------------------------------------
    # 2. IMPORTANT:
    # Load ALL checkpoint results, not only newly evaluated ones
    # ---------------------------------------------------------

    results_df = load_checkpoint_results()

    if results_df.empty:
        raise RuntimeError(
            "No valid evaluation results were found in the checkpoint."
        )

    # ---------------------------------------------------------
    # 3. Make sure the expected columns exist
    # ---------------------------------------------------------

    required_columns = [
        "category",
        "recall@10",
        "mrr",
        "ndcg@10",
        "absolute_error",
        "percentage_error",
        "within_20",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in results_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "The checkpoint contains results from an older evaluation schema.\n\n"
            f"Missing columns: {missing_columns}\n\n"
            "Delete eval_checkpoint.jsonl and run the evaluation again."
        )

    # ---------------------------------------------------------
    # 4. Retrieval category data
    # ---------------------------------------------------------

    for _, row in results_df.iterrows():

        category = row["category"]

        recall_value = row["recall@10"]

        if pd.notna(recall_value):
            category_recall[category].append(recall_value)

        if pd.notna(row["absolute_error"]):
            category_mae[category].append(
                row["absolute_error"]
            )

    # ---------------------------------------------------------
    # 5. Valid price predictions
    # ---------------------------------------------------------

    valid = results_df.dropna(
        subset=[
            "absolute_error",
            "percentage_error"
        ]
    )

    # ---------------------------------------------------------
    # 6. Retrieval summary
    # ---------------------------------------------------------

    retrieval_html = f"""
    <div style="padding: 0;">

        {format_metric_html(
            "Recall@10",
            results_df["recall@10"].mean(),
            "recall"
        )}

        {format_metric_html(
            "MRR",
            results_df["mrr"].mean(),
            "mrr"
        )}

        {format_metric_html(
            "nDCG@10",
            results_df["ndcg@10"].mean(),
            "ndcg"
        )}

        <div style="
            margin-top: 15px;
            padding: 10px;
            background-color: #e7f3ff;
            border-radius: 5px;
            font-size: 13px;
            color: #333;
        ">
            Relevance = same category + price within
            {int(PRICE_TOLERANCE * 100)}% tolerance.
        </div>

        <div style="
            margin-top: 10px;
            padding: 10px;
            background-color: #d4edda;
            border-radius: 5px;
            text-align: center;
            border: 1px solid #c3e6cb;
        ">
            <span style="
                font-size: 14px;
                color: #155724;
                font-weight: bold;
            ">
                ✓ Evaluation Complete:
                {len(results_df)} / {len(golden_dataset)} tests
            </span>
        </div>

    </div>
    """

    # ---------------------------------------------------------
    # 7. Retrieval chart
    # ---------------------------------------------------------

    retrieval_chart_df = pd.DataFrame([
        {
            "Category": category,
            "Average Recall@10": sum(values) / len(values)
        }
        for category, values in category_recall.items()
        if values
    ])

    # ---------------------------------------------------------
    # 8. Price summary
    # ---------------------------------------------------------

    if len(valid) > 0:

        mae = valid["absolute_error"].mean()

        rmse = np.sqrt(
            np.mean(
                valid["absolute_error"] ** 2
            )
        )

        within_20 = valid["within_20"].mean()

    else:

        mae = np.nan
        rmse = np.nan
        within_20 = np.nan

    extraction_rate = (
        len(valid) / len(results_df)
        if len(results_df) > 0
        else 0
    )

    price_html = f"""
    <div style="padding: 0;">

        {format_metric_html(
            "MAE",
            mae,
            "mae",
            is_currency=True
        )}

        {format_metric_html(
            "RMSE",
            rmse,
            "rmse",
            is_currency=True
        )}

        {format_metric_html(
            "Within +/-20%",
            within_20,
            "within20",
            is_percentage=True
        )}

        {format_metric_html(
            "Answer extraction rate",
            extraction_rate,
            "extraction",
            is_percentage=True
        )}

    </div>
    """

    # ---------------------------------------------------------
    # 9. Price category chart
    # ---------------------------------------------------------

    mae_chart_df = pd.DataFrame([
        {
            "Category": category,
            "Average MAE ($)": sum(values) / len(values)
        }
        for category, values in category_mae.items()
        if values
    ])

    # ---------------------------------------------------------
    # 10. Return dashboard outputs
    # ---------------------------------------------------------

    return (
        retrieval_html,
        retrieval_chart_df,
        price_html,
        mae_chart_df,
    )


def main():
    theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])
    with gr.Blocks(title="RAG Evaluation Dashboard - New Path") as app:
        gr.Markdown("# 📊 RAG Evaluation Dashboard (New Path / Pricer)")
        gr.Markdown(
            "Retrieval relevance = same category + price within tolerance "
            "(golden-set products are held out of the KB by design, see note above)."
        )

        gr.Markdown("## 🔍 Retrieval Evaluation")
        run_button = gr.Button("Run Full Evaluation", variant="primary", size="lg")
        with gr.Row():
            with gr.Column(scale=1):
                retrieval_metrics = gr.HTML("<div style='padding:20px;text-align:center;color:#999;'>Click 'Run Full Evaluation' to start</div>")
            with gr.Column(scale=1):
                retrieval_chart = gr.BarPlot(x="Category", y="Average Recall@10", title="Average Recall@10 by Category", y_lim=[0, 1], height=350)

        gr.Markdown("## 💲 Price Estimation Evaluation")
        with gr.Row():
            with gr.Column(scale=1):
                price_metrics = gr.HTML("<div style='padding:20px;text-align:center;color:#999;'>Run evaluation above first</div>")
            with gr.Column(scale=1):
                price_chart = gr.BarPlot(x="Category", y="Average MAE ($)", title="Average MAE ($) by Category", height=350)

        run_button.click(
            fn=run_evaluation,
            outputs=[retrieval_metrics, retrieval_chart, price_metrics, price_chart],
        )

    app.launch(inbrowser=True, theme=theme)


if __name__ == "__main__":
    main()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Resuming: 200 tests already done, skipping them.
Resuming: 200 tests already done, skipping them.
Resuming: 200 tests already done, skipping them.
